# JC — Holy OG GPU Colab Factory

One-click entry point for the `thebigostreets-byte/jc-the-holy-og` project.

**What this notebook does:** mounts Google Drive, keeps a persistent JC workspace there, syncs the current `main` branch, detects the NVIDIA GPU, installs the project dependencies, and provides a browser preview.

The Colab GPU is used for Python/AI/asset-processing workloads. The Three.js game itself still renders in the browser.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

JC_DRIVE = '/content/drive/MyDrive/JC/Holy-OG'
import os
os.makedirs(JC_DRIVE, exist_ok=True)
print('JC Drive workspace:', JC_DRIVE)

In [ ]:
# 2. Sync the real JC repository into Drive
import os, subprocess, textwrap
REPO = 'https://github.com/thebigostreets-byte/jc-the-holy-og.git'
REPO_DIR = os.path.join(JC_DRIVE, 'repo')

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', 'main', REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)

print(subprocess.check_output(['git','-C',REPO_DIR,'log','-1','--oneline'], text=True))

In [ ]:
# 3. Detect the Colab NVIDIA GPU
import subprocess, os, sys
try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as e:
    raise RuntimeError('No NVIDIA GPU runtime is attached. In Colab choose Runtime > Change runtime type > GPU.') from e

In [ ]:
# 4. Persistent Python environment + common JC asset tooling
import os, subprocess, sys
VENV = os.path.join(JC_DRIVE, 'venv')
if not os.path.exists(os.path.join(VENV, 'bin', 'python')):
    subprocess.run([sys.executable, '-m', 'venv', VENV], check=True)

PIP = os.path.join(VENV, 'bin', 'pip')
subprocess.run([PIP, 'install', '-U', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([PIP, 'install', 'numpy', 'pillow', 'trimesh', 'pygltflib'], check=True)

# Install the JS dependencies from the actual JC package when present.
if os.path.exists(os.path.join(REPO_DIR, 'package.json')):
    subprocess.run(['npm', 'install'], cwd=REPO_DIR, check=True)

print('Persistent JC environment ready.')

In [ ]:
# 5. GPU verification for Python workloads
import sys, subprocess
PY = os.path.join(VENV, 'bin', 'python')
subprocess.run([PY, '-c', "import sys; print('Python:',sys.version); import torch; print('PyTorch:',torch.__version__); print('CUDA:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"], check=False)

## 6. JC browser preview

Run this cell only when you want the current JC web build available from the Colab session. The browser renderer remains local to the browser; Colab is not replacing WebGL rendering with CUDA.

In [ ]:
# Start Vite in the background and expose it through Colab's built-in proxy.
import subprocess, time, os
log = open(os.path.join(JC_DRIVE, 'vite.log'), 'w')
proc = subprocess.Popen(['npm','run','dev','--','--host','0.0.0.0','--port','5173'], cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT)
time.sleep(4)
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(5173)")
print('JC preview:', url)
print('Vite PID:', proc.pid)

## Persistent locations

- `My Drive/JC/Holy-OG/repo` — synced JC source
- `My Drive/JC/Holy-OG/venv` — persistent Python environment
- `My Drive/JC/Holy-OG/assets` — generated assets
- `My Drive/JC/Holy-OG/checkpoints` — model/checkpoint files
- `My Drive/JC/Holy-OG/outputs` — generated outputs
- `My Drive/JC/Holy-OG/logs` — logs
